# Phase 6 quick prediction sweep

No GPU or retraining. Mount the same Drive, read existing prediction ZIPs, create short variants and a conservative consensus blend. **Score improvements are not guaranteed. Keep the 0.811055 baseline submission.** These are experimental Codabench candidates and do not replace course architecture requirements.


In [ ]:
from google.colab import drive
from pathlib import Path
import json, re, zipfile, statistics

drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/CS781_PHASE6_OUTPUT")
DEST = ROOT / "P6_QUICK_20MIN"
DEST.mkdir(parents=True, exist_ok=True)
RUNS = ("P6_E23_dense_full", "P6_E23_long_partial_448",
        "P6_E23_full_384", "P6_E23_head_512", "P6_E23_salient_448")

def read_run(run_id):
    archive_path = ROOT / run_id / "submission.zip"
    if not archive_path.exists():
        print("Missing, skipping:", archive_path)
        return None
    with zipfile.ZipFile(archive_path) as archive:
        assert archive.namelist() == ["prediction.json"]
        rows = json.loads(archive.read("prediction.json"))
    assert len(rows) == 100
    assert len({str(row["id"]) for row in rows}) == 100
    assert all(isinstance(row["summary"], str) and row["summary"].strip() for row in rows)
    return rows

loaded = {run: rows for run in RUNS if (rows := read_run(run)) is not None}
assert "P6_E23_dense_full" in loaded, "The scored 0.811 baseline ZIP is required"
ids = [str(row["id"]) for row in loaded["P6_E23_dense_full"]]
for run, rows in loaded.items():
    assert [str(row["id"]) for row in rows] == ids, f"ID order mismatch: {run}"
    lengths = [len(row["summary"].split()) for row in rows]
    print(run, "average words:", round(statistics.mean(lengths), 1))


In [ ]:
def sentence_trim(text, max_words=230):
    parts = re.split(r"(?<=[.!?])\s+", text.strip())
    chosen = []
    length = 0
    for part in parts:
        count = len(part.split())
        if length + count > max_words:
            break
        chosen.append(part)
        length += count
    if length >= 120:
        return " ".join(chosen).strip()
    return " ".join(text.split()[:max_words]).strip()

def words(text):
    return {w for w in re.findall(r"[a-zA-Z]{3,}|[0-9]+", text.lower())
            if w not in {"the", "and", "for", "that", "with", "this", "from",
                         "was", "were", "are", "have", "has", "court", "case"}}

def jaccard(a, b):
    return len(a & b) / max(1, len(a | b))

def consensus_extension(index, base):
    candidates = []
    base_words = words(base)
    for run, rows in loaded.items():
        if run == "P6_E23_dense_full":
            continue
        for sentence in re.split(r"(?<=[.!?])\s+", rows[index]["summary"]):
            count = len(sentence.split())
            if not 12 <= count <= 55:
                continue
            terms = words(sentence)
            if not terms or jaccard(terms, base_words) >= 0.45:
                continue
            candidates.append((run, sentence.strip(), terms))
    best = None
    for i, (run, sentence, terms) in enumerate(candidates):
        agreement = max((jaccard(terms, other_terms)
                         for other_run, _, other_terms in candidates[i+1:]
                         if other_run != run), default=0)
        if agreement >= 0.35 and (best is None or agreement > best[0]):
            best = (agreement, sentence)
    return best[1] if best else ""

def write_zip(name, rows):
    assert len(rows) == 100
    assert [str(row["id"]) for row in rows] == ids
    assert all(row["summary"].strip() for row in rows)
    target = DEST / (name + ".zip")
    data = json.dumps(rows, ensure_ascii=False, separators=(",", ":")) + "\n"
    with zipfile.ZipFile(target, "w", zipfile.ZIP_DEFLATED) as archive:
        archive.writestr("prediction.json", data)
    with zipfile.ZipFile(target) as archive:
        assert archive.namelist() == ["prediction.json"]
        assert archive.testzip() is None
    print("READY:", target)

for run, rows in loaded.items():
    if run == "P6_E23_dense_full":
        continue
    trimmed = [{"id": str(row["id"]), "summary": sentence_trim(row["summary"], 230)}
               for row in rows]
    write_zip("trim230_" + run, trimmed)

baseline = loaded["P6_E23_dense_full"]
blended = []
changed = 0
for index, row in enumerate(baseline):
    extra = consensus_extension(index, row["summary"])
    if extra:
        changed += 1
    blended.append({"id": str(row["id"]),
                    "summary": (row["summary"].strip() + " " + extra).strip()})
if changed:
    write_zip("baseline_plus_one_consensus_sentence", blended)
print("Consensus rows changed:", changed, "of 100")
print("Your scored 0.811 baseline ZIP remains untouched.")
